In [ ]:
import os, requests, csv, time, math
from datetime import datetime

In [ ]:
# INSERT GITHUB TOKEN
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
REPO_NAME = "talonhub/community"
ACCEPTED_STATUS_CODE = 200
url = f"https://api.github.com/repos/{REPO_NAME}/pulls"
TOTAL_PRS = 1564
PER_PAGE = 100
pages_needed = math.ceil(TOTAL_PRS / PER_PAGE)

all_prs = []
headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
for page in range(1, pages_needed + 1):
    params = {
        'state': 'closed',
        'sort': 'created',
        'direction': 'desc',
        'per_page': TOTAL_PRS,
        'page': page
    }
    response = requests.get(url, headers=headers, params=params)
    if response.status_code != ACCEPTED_STATUS_CODE:
        break
    page_prs = response.json()
    if not page_prs:
        break
    all_prs.extend(page_prs)


In [ ]:
def get_comment_number(pr_comment_url):
    time.sleep(0.05)
    comment_resp = requests.get(pr_comment_url, headers=headers, params={'per_page': TOTAL_PRS})
    if comment_resp.status_code == ACCEPTED_STATUS_CODE:
        return comment_resp.json()
    else:
        print("Error Connecting")
        return []
    
def get_reviews(pr_reviews_url):
    time.sleep(0.05)
    reviews_resp = requests.get(pr_reviews_url, headers=headers, params={'per_page': TOTAL_PRS})
    if reviews_resp.status_code == ACCEPTED_STATUS_CODE:
        return reviews_resp.json()
    else:
        print("Error Connecting")
        return []

def check_user(comment):
    return comment["user"]["type"].lower() == "bot"

def check_maintainer(comment):
    MAINTAINER_ROLES = ("OWNER", "MEMBER", "COLLABORATOR")
    return comment["author_association"] in MAINTAINER_ROLES


In [ ]:
with open('Full_Community_MSR.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([['PR Number', 'Author', 'Created Time', 'Merged', 'Time to Merge (hours)', 'Review Comments URL', 'Comments URL','Maintainer Comments', 'Contributor Comments']])
    for i, pr in enumerate(all_prs):
        pr_number = pr['number']
        pr_author = pr['user']['login']
        created_time = pr['created_at']
        merged = pr['merged_at'] is not None
        review_commits = pr['review_comments_url']
        comments_url = pr['comments_url']

        if merged:
            time_to_merge = (datetime.fromisoformat(pr['merged_at'].replace("Z", "+00:00")) - datetime.fromisoformat(created_time.replace("Z", "+00:00"))).total_seconds() / 3600
        else:
            time_to_merge = 0

        issue_comments = get_comment_number(comments_url)
        maintainer_comments = []
        contributor_comments = []
        if issue_comments != 0:
            for comment in issue_comments:
                if check_user(comment):
                    continue
                if check_maintainer(comment):
                    maintainer_comments.append(comment)
                else:
                    contributor_comments.append(comment)
        else:
            print(f"No Comments for {pr_number}")

        writer.writerow([[pr_number, pr_author, created_time, merged, f"{time_to_merge:.2f}" if merged else 0, review_commits, comments_url, maintainer_comments, contributor_comments]])
        # if i == 10:
        #     break

dict_keys(['url', 'pull_request_review_id', 'id', 'node_id', 'diff_hunk', 'path', 'commit_id', 'original_commit_id', 'user', 'body', 'created_at', 'updated_at', 'html_url', 'pull_request_url', '_links', 'reactions', 'start_line', 'original_start_line', 'start_side', 'line', 'original_line', 'side', 'author_association', 'original_position', 'position', 'subject_type'])